**Autor:** Andrés Alejandro Rodríguez Lozano  
**Portafolio:** Portafolio de Andrés Alejandro Rodríguez Lozano

# Capítulo 6 — Activos: riesgo vs retorno

Scatter de volatilidad anualizada vs retorno anualizado por ticker (tamaño ≈ peso eToro).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({
    'figure.dpi': 72,
    'savefig.dpi': 72,
    'figure.figsize': (8, 3.8),
    'font.size': 9,
    'svg.fonttype': 'none',
    'path.simplify': True,
    'path.simplify_threshold': 0.2,
})

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import portfolio_utils as pu

def load_monthly():
    """Carga precios mensuales: cache mensual, diario, o yfinance."""
    mpath = ROOT / 'data' / 'precios_mensuales.csv'
    if mpath.exists():
        print('Precios mensuales (cache):', mpath.name)
        return pd.read_csv(mpath, index_col=0, parse_dates=True).loc[: '2026-09-30']
    csv = ROOT / 'data' / 'precios_ajustados_diarios.csv'
    if csv.exists():
        daily = pd.read_csv(csv, index_col=0, parse_dates=True)
        print('Precios diarios (cache):', csv.name)
        return pu.to_month_end(daily).loc[: '2026-09-30']
    tickers = sorted(set(list(pu.CURRENT_WEIGHTS) + [pu.BENCHMARK]))
    print('Descargando precios con yfinance…')
    daily = pu.download_prices(tickers)
    return pu.to_month_end(daily).loc[: '2026-09-30']


monthly = load_monthly()
# Necesitamos un mes previo a ene-2016 para el primer retorno
px = monthly.loc[: '2026-09-30']
rets = px.pct_change().loc['2016-01-31':'2026-09-30'].dropna(how='any')
tickers = list(pu.CURRENT_WEIGHTS) + [pu.BENCHMARK]
mu = rets[tickers].mean() * 12
sig = rets[tickers].std() * (12 ** 0.5)
print(pd.DataFrame({'ret': mu, 'vol': sig}).map(lambda x: f'{100*x:.2f}%'))


In [ ]:
# Scatter riesgo–retorno
fig, ax = plt.subplots(figsize=(7, 5))
for t in tickers:
    size = 80 if t == pu.BENCHMARK else 200 * pu.CURRENT_WEIGHTS.get(t, 0.05)
    ax.scatter(100 * sig[t], 100 * mu[t], s=size, alpha=0.8)
    ax.annotate(t, (100 * sig[t], 100 * mu[t]), xytext=(5, 4), textcoords='offset points', fontsize=8)
ax.set_xlabel('Volatilidad anualizada (%)')
ax.set_ylabel('Retorno anualizado (%)')
ax.set_title('Riesgo–retorno por activo')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Conclusiones
1. SMH concentra retorno y riesgo; IEMG aportó menos en 2016–2024.

## Ejercicios
1. Sharpe por activo (rf≈T-Bill).
2. Excluye SMH del scatter.
